# Plain-Language Notices — LoRA fine-tune (Colab/Kaggle GPU)

Fine-tunes a small instruct model to rewrite bureaucratic benefit notices into plain
language while preserving operative facts. Uses Unsloth + TRL (QLoRA). Runtime: a free
Colab T4 is enough. Settings mirror `train/config.yaml`.

**Before running:** set the runtime to GPU, and make `data/pairs.jsonl` available (clone
the repo, or upload the file).

In [ ]:
# 1. Install (Colab/Kaggle)
%pip install -q unsloth "trl>=0.9" "peft>=0.11" "transformers>=4.44" "datasets>=2.20"

In [ ]:
# 2. Get the training data (clone the repo or upload data/pairs.jsonl)
import os
if not os.path.exists('data/pairs.jsonl'):
    !git clone -q https://github.com/adamselzer/plain-language-notices.git repo
    !cp -r repo/data .
print('train pairs:', sum(1 for _ in open('data/pairs.jsonl')))

## Load the base model in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=13,
)

## Format the pairs as chat examples
Each example is (system, user=bureaucratic notice, assistant=plain rewrite), rendered
with the model's chat template. System + user mirror `src/rewrite.py` so training and
inference match.

In [ ]:
import json
from datasets import Dataset

SYSTEM = ('You rewrite government benefit notices into plain language at a sixth-grade '
          'reading level, preserving every operative fact (amounts, dates, the action, '
          'the reason, and appeal/hearing rights with the deadline).')

rows = [json.loads(l) for l in open('data/pairs.jsonl') if l.strip()]
def to_text(r):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':'Rewrite this benefit notice in plain language:\n\n'+r['original']},
            {'role':'assistant','content':r['target']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)
ds = Dataset.from_list([{'text': to_text(r)} for r in rows])
print(ds[0]['text'][:400])

## Train (TRL SFTTrainer + SFTConfig)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=3, learning_rate=2e-4, warmup_steps=5,
        logging_steps=5, seed=13, output_dir='outputs', optim='adamw_8bit',
    ),
)
trainer.train()

## Save the adapter
Download `adapters/plain-notices-lora/` and place it in the repo so `src/rewrite.py`
and `eval/run_eval.py --with-finetuned` can load it.

In [ ]:
model.save_pretrained('adapters/plain-notices-lora')
tokenizer.save_pretrained('adapters/plain-notices-lora')
print('saved adapter')

## Quick sanity check

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role':'system','content':SYSTEM},
        {'role':'user','content':'Rewrite this benefit notice in plain language:\n\n'
         'This correspondence constitutes formal notification that your application has been '
         'denied, predicated upon excess income. You may request a hearing within 90 days.'}]
inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=200)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))